In [3]:
import os
import numpy as np
import pandas as pd

current_dir = os.getcwd()
folder_path = os.path.join(current_dir, "..", "data_raw")

# ĐỌC VÀ GỘP DỮ LIỆU (MERGE)
if not os.path.exists(folder_path):
    print(f"Lỗi: Không tìm thấy thư mục {folder_path}")

else:
    file_list = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
    file_list.sort()

    all_df = []

    for file_name in file_list:
        full_path = os.path.join(folder_path, file_name)

        temp_df = pd.read_csv(full_path)

        # Chuẩn hóa tên cột
        temp_df.columns = [
            c.lower().strip().replace(" ", "_")
            for c in temp_df.columns
        ]

        all_df.append(temp_df)

    df = pd.concat(all_df, ignore_index=True)
    df = df.drop_duplicates()
    print(
        f"--- Đã gộp xong {len(file_list)} file. Tổng số dòng: {len(df)} ---"
    )

    # Xóa duplicate theo local_time, giữ hàng đầu tiên
    df = df.drop_duplicates(subset=['local_time'], keep='first')
    print(f"Sau khi xóa duplicate theo giờ: {len(df):,} hàng")

    # HÀM CHUYỂN PM2.5 -> AQI
    def pm25_to_aqi(pm):
        if pd.isna(pm):
            return np.nan

        pm = float(pm)

        if pm <= 12.0:
            return (50 / 12.0) * pm
        elif pm <= 35.4:
            return 51 + (pm - 12.1) * (49 / (35.4 - 12.1))
        elif pm <= 55.4:
            return 101 + (pm - 35.5) * (49 / (55.4 - 35.5))
        elif pm <= 150.4:
            return 151 + (pm - 55.5) * (99 / (150.4 - 55.5))
        elif pm <= 250.4:
            return 201 + (pm - 150.5) * (99 / (250.4 - 150.5))
        elif pm <= 350.4:
            return 301 + (pm - 250.5) * (99 / (350.4 - 250.5))
        elif pm <= 500.4:
            return 401 + (pm - 350.5) * (99 / (500.4 - 350.5))
        else:
            return 500

    # HÀM GÁN NHÃN AQI
    def get_aqi_label(v):
        if pd.isna(v):
            return np.nan

        if v <= 50:
            return "Good"
        elif v <= 100:
            return "Moderate"
        elif v <= 150:
            return "Unhealthy for sensitive groups"
        elif v <= 200:
            return "Unhealthy"
        elif v <= 300:
            return "Very Unhealthy"
        else:
            return "Hazardous"
        
    # FEATURE ENGINEERING
    df["local_time"] = pd.to_datetime(df["local_time"])

    df["date"] = df["local_time"].dt.date
    df["year"] = df["local_time"].dt.year
    df["month"] = df["local_time"].dt.month
    df["hour"] = df["local_time"].dt.hour

    df["day_of_week"] = df["local_time"].dt.dayofweek

    df["is_weekend"] = df["day_of_week"].apply(
        lambda x: 1 if x >= 5 else 0
    )

    df["is_rush_hour"] = df["hour"].apply(
        lambda x: 1 if x in [6, 7, 8, 9, 17, 18, 19, 20] else 0
    )

    def get_season(m):
        if m in [12, 1, 2]:
            return 0  # Đông
        elif m in [3, 4, 5]:
            return 1  # Xuân
        elif m in [6, 7, 8]:
            return 2  # Hạ
        else:
            return 3  # Thu

    df["season"] = df["month"].apply(get_season)

    # XỬ LÝ MISSING VALUES
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    df[numeric_cols] = df[numeric_cols].interpolate(
        method="linear",
        limit_direction="both"
    )

    df[numeric_cols] = df[numeric_cols].ffill().bfill()

    # XỬ LÝ OUTLIERS
    # Winsorize theo percentile thay vì clip cứng
    df["co"] = df["co"].clip(lower=0, upper=1500)
    for col in ["no2", "o3", "pm10", "so2"]:
        if col in df.columns:
            p995 = df[col].quantile(0.995)
            df[col] = df[col].clip(lower=0, upper=p995)


    # XÓA CÁC CỘT KHÔNG CẦN THIẾT
    columns_to_drop = ["utc_time", "city", "timezone", "country_code"]
    df = df.drop(columns=columns_to_drop, errors="ignore")

    # ĐƯA TIME FEATURES LÊN ĐẦU
    time_feat = [
        "local_time",
        "date",
        "year",
        "month",
        "hour",
        "day_of_week",
        "is_weekend",
        "is_rush_hour",
        "season",
    ]

    other_cols = [
        c for c in df.columns
        if c not in time_feat
    ]

    df = df[time_feat + other_cols]

    # Winsorize: thay các giá trị > percentile 99.5% bằng chính ngưỡng đó
    p995_pm25 = df["pm25"].quantile(0.99)
    df["pm25"] = df["pm25"].clip(upper=p995_pm25)

    # Phát hiện hàng lỗi
    aqi_equals_pm25  = (df["aqi"] - df["pm25"]).abs() < 5      # AQI == PM2.5
    aqi_500_low_pm25 = (df["aqi"] >= 500) & (df["pm25"] < 100)  # AQI=500 nhưng PM2.5 thấp
    aqi_over_p995    = df["aqi"] > df["aqi"].quantile(0.995)     # AQI quá cực đoan

    error_mask = aqi_equals_pm25 | aqi_500_low_pm25 | aqi_over_p995

    print(f"Phát hiện {error_mask.sum()} hàng lỗi AQI → tính lại từ PM2.5")
    print(f"Giữ nguyên {(~error_mask).sum()} hàng AQI gốc")

    # Chỉ tính lại AQI cho hàng lỗi
    df.loc[error_mask, "aqi"] = df.loc[error_mask, "pm25"].apply(pm25_to_aqi)

    # GÁN NHÃN AQI
    df["aqi"] = df["aqi"].round(0).astype(int)
    if "aqi" in df.columns:
        df["aqi_category"] = df["aqi"].apply(get_aqi_label)

    df.to_csv(
        "hanoi_aqi_cleaned.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("\n--- HOÀN THÀNH CLEAN DATA THÀNH CÔNG ---")

    print(
        f"Số lượng giá trị thiếu còn lại: "
        f"{df.isnull().sum().sum()}"
    )
    print(f"Số lượng cột hiện có: {len(df.columns)}")
    print(f"📈 AQI cao nhất: {df['aqi'].max():.1f}")
    print(f"📈 PM2.5 cao nhất: {df['pm25'].max():.1f}")
    print(f"📈 CO cao nhất: {df['co'].max():.1f}")
    print(f"📈 NO2 cao nhất: {df['no2'].max():.1f}")
    print(f"📈 O3 cao nhất: {df['o3'].max():.1f}")
    print(f"📈 PM10 cao nhất: {df['pm10'].max():.1f}")
    print(f"📈 SO2 cao nhất: {df['so2'].max():.1f}")

--- Đã gộp xong 4 file. Tổng số dòng: 30341 ---
Sau khi xóa duplicate theo giờ: 30,336 hàng
Phát hiện 818 hàng lỗi AQI → tính lại từ PM2.5
Giữ nguyên 29518 hàng AQI gốc

--- HOÀN THÀNH CLEAN DATA THÀNH CÔNG ---
Số lượng giá trị thiếu còn lại: 0
Số lượng cột hiện có: 24
📈 AQI cao nhất: 325.0
📈 PM2.5 cao nhất: 212.0
📈 CO cao nhất: 1500.0
📈 NO2 cao nhất: 185.0
📈 O3 cao nhất: 211.0
📈 PM10 cao nhất: 396.9
📈 SO2 cao nhất: 269.5
